# 06. Configuración y conexión con MongoDB Atlas

Este notebook configura y valida la conexión entre el proyecto y MongoDB Atlas mediante el driver oficial PyMongo.

MongoDB se utilizará posteriormente como capa documental para almacenar resúmenes diarios de las mediciones solares, referencias a gráficas, información visual y resúmenes de resultados de los modelos. Los datos tabulares detallados y las predicciones a nivel de registro permanecerán almacenados en PostgreSQL.

En esta primera fase únicamente se comprueba la conexión con Atlas y se prepara el acceso seguro mediante variables de entorno.

## Importación de dependencias y variables de entorno

Las credenciales de conexión se almacenan en un archivo `.env` excluido del control de versiones. De esta forma, el notebook puede conectarse a MongoDB Atlas sin incluir información sensible directamente en el código.

In [6]:
import os

from dotenv import load_dotenv
from pymongo import MongoClient
from pymongo.server_api import ServerApi

In [7]:
load_dotenv()

mongodb_uri = os.getenv("MONGODB_URI")
mongodb_database = os.getenv("MONGODB_DATABASE")

if not mongodb_uri:
    raise ValueError(
        "No se ha encontrado la variable MONGODB_URI en el archivo .env"
    )

if not mongodb_database:
    raise ValueError(
        "No se ha encontrado la variable MONGODB_DATABASE en el archivo .env"
    )

print("Variables de entorno cargadas correctamente.")
print(f"Base de datos configurada: {mongodb_database}")

Variables de entorno cargadas correctamente.
Base de datos configurada: solar_irradiance_db


## Creación del cliente de MongoDB

Se crea un cliente mediante PyMongo y se utiliza la Stable API de MongoDB. A continuación, se envía una operación `ping` para verificar que el clúster está accesible y que las credenciales son válidas.

In [8]:
client = MongoClient(
    mongodb_uri,
    server_api=ServerApi("1"),
    serverSelectionTimeoutMS=10_000,
)

try:
    client.admin.command("ping")
    print("Conexión correcta con MongoDB Atlas.")
except Exception as exc:
    print("No se ha podido conectar con MongoDB Atlas.")
    raise exc

Conexión correcta con MongoDB Atlas.


In [9]:
db = client[mongodb_database]

print(f"Base de datos seleccionada: {db.name}")

Base de datos seleccionada: solar_irradiance_db


In [10]:
database_names = client.list_database_names()

print("Bases de datos accesibles:")
for database_name in database_names:
    print(f"- {database_name}")

Bases de datos accesibles:
- admin
- local


## Resultado de la configuración

La conexión con MongoDB Atlas se ha verificado correctamente mediante PyMongo. También se ha seleccionado la base de datos lógica `solar_irradiance_db`, que se utilizará en los siguientes notebooks para almacenar documentos diarios.

En este punto todavía no se han creado colecciones ni insertado documentos. La creación efectiva de la base de datos se realizará al cargar el primer documento de prueba.

In [11]:
client.close()

print("Conexión con MongoDB Atlas cerrada correctamente.")

Conexión con MongoDB Atlas cerrada correctamente.
